# 05 – Gradio Demo (Playlist Optimizer)
**Project**: Playlist Optimizer  
**Author**: S33mi  

Interactive demo:
- Search a seed track
- Choose playlist length
- Optimize with Simulated Annealing (fast local search)
- See score breakdown + energy/tempo curve

## 1. Install & Imports
```bash
pip install gradio
```


In [8]:
import os
from io import BytesIO
import random
import math
from copy import deepcopy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import joblib
import gradio as gr
from sklearn.metrics.pairwise import cosine_similarity
import warnings

warnings.filterwarnings("ignore")
print("Libraries loaded.")


Libraries loaded.


## 2. Load Artifacts via URL

In [9]:
BASE = "https://raw.githubusercontent.com/S33mi/spotify-music-recommender/main"

URL_PART1 = f"{BASE}/data/processed/spotify_with_clusters_part1.csv"
URL_PART2 = f"{BASE}/data/processed/spotify_with_clusters_part2.csv"
URL_X     = f"{BASE}/data/processed/X_scaled.npy"
URL_MOOD  = f"{BASE}/data/models/mood_labels.pkl"

def download(url: str) -> bytes:
    print(f"Downloading {url} ...")
    r = requests.get(url, timeout=120)
    r.raise_for_status()
    return r.content

print("Loading data ...")
p1 = download(URL_PART1)
p2 = download(URL_PART2)
with open("_part1.tmp", "wb") as f: f.write(p1)
with open("_part2.tmp", "wb") as f: f.write(p2)
df = pd.concat([pd.read_csv("_part1.tmp"), pd.read_csv("_part2.tmp")], ignore_index=True)
os.remove("_part1.tmp"); os.remove("_part2.tmp")

X = np.load(BytesIO(download(URL_X)))
mood_labels = joblib.load(BytesIO(download(URL_MOOD)))
if "mood" not in df.columns and "cluster" in df.columns:
    df["mood"] = df["cluster"].map(mood_labels)

assert len(df) == len(X)
print(f"Ready – {len(df)} tracks loaded.")


Loading data ...
Ready – 89741 tracks loaded.


### Pre Processing

In order to avoid `NaN` value data in selected playlist do the same for GA (if required)

In [10]:
# -------------------------------------------------
# Clean the dataset
# -------------------------------------------------
print("Original shape:", df.shape)

# 1. Remove rows with missing critical information
critical_cols = ["track_name", "artists", "track_genre"]
df = df.dropna(subset=critical_cols)

# 2. Fill missing numerical values
for col in ["energy", "tempo", "popularity"]:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

# 3. Reset index so iloc works perfectly (very important!)
df = df.reset_index(drop=True)

# 4. Keep X aligned with the cleaned df
# (Assuming X was loaded in the same order as original df)
X = X[df.index] if len(X) != len(df) else X   # safe alignment

print("Cleaned shape:", df.shape)
print("Remaining NaNs in critical columns:")
print(df[critical_cols].isna().sum())

Original shape: (89741, 23)
Cleaned shape: (45081, 23)
Remaining NaNs in critical columns:
track_name     0
artists        0
track_genre    0
dtype: int64


## 3. Scoring + SA (lightweight versions)


In [11]:
def smoothness_score(energy, tempo):
    if len(energy) < 2:
        return 1.0
    e_jump = np.abs(np.diff(energy)).mean()
    t_jump = (np.abs(np.diff(tempo)) / 50.0).mean()
    return 1.0 / (1.0 + 0.6 * e_jump + 0.4 * t_jump)

def total_score(seed_idx, indices):
    energy = df.iloc[indices]["energy"].values
    tempo  = df.iloc[indices]["tempo"].values
    sims = cosine_similarity(X[seed_idx].reshape(1, -1), X[indices])[0]
    s_sim = float(sims.mean())
    s_smooth = smoothness_score(energy, tempo)
    # diversity
    if len(indices) > 1:
        sim_m = cosine_similarity(X[indices])
        triu = sim_m[np.triu_indices(len(indices), k=1)]
        s_div = float(1.0 - triu.mean())
    else:
        s_div = 1.0
    # mood
    if "mood" in df.columns:
        seed_mood = df.iloc[seed_idx]["mood"]
        s_mood = float((df.iloc[indices]["mood"] == seed_mood).mean())
    else:
        s_mood = 0.5
    total = 0.30*s_smooth + 0.30*s_sim + 0.25*s_div + 0.15*s_mood
    return total, {"smoothness": s_smooth, "similarity": s_sim, "diversity": s_div, "mood": s_mood}

def nn_playlist(seed_idx, length):
    sims = cosine_similarity(X[seed_idx].reshape(1, -1), X)[0]
    sims[seed_idx] = -np.inf
    return np.argsort(sims)[-length:][::-1].tolist()

def neighbor(state, seed_idx, n_tracks):
    s = state[:]
    length = len(s)
    move = random.random()
    if move < 0.35:
        i, j = random.sample(range(length), 2)
        s[i], s[j] = s[j], s[i]
    elif move < 0.65:
        i, j = random.sample(range(length), 2)
        track = s.pop(i)
        s.insert(j, track)
    else:
        pos = random.randrange(length)
        used = set(s)
        candidates = [t for t in range(n_tracks) if t != seed_idx and t not in used]
        if candidates:
            s[pos] = random.choice(candidates)
    return s

def run_sa(seed_idx, length, steps=2500, T0=0.12, alpha=0.995):
    """Fast SA for the demo."""
    n_tracks = len(df)
    current = nn_playlist(seed_idx, length)
    cur_score, _ = total_score(seed_idx, current)
    best, best_score = current[:], cur_score
    T = T0
    for _ in range(steps):
        cand = neighbor(current, seed_idx, n_tracks)
        cand_score, _ = total_score(seed_idx, cand)
        delta = cur_score - cand_score   # we maximize score
        if delta < 0 or random.random() < math.exp(-delta / max(T, 1e-9)):
            current, cur_score = cand, cand_score
            if cur_score > best_score:
                best, best_score = current[:], cur_score
        T *= alpha
    return best, best_score

## 4. Gradio Helper Functions


In [12]:
def search_tracks(query, top_n=8):
    if not query or len(query.strip()) < 2:
        return gr.update(choices=[], value=None)
    q = query.lower().strip()
    mask = (
        df["track_name"].str.lower().str.contains(q, na=False)
        | df["artists"].str.lower().str.contains(q, na=False)
    )
    results = df.loc[mask, ["track_id", "track_name", "artists"]].head(top_n)
    choices = [
        f"{r.track_name} — {r.artists} ({r.track_id})"
        for _, r in results.iterrows()
    ]
    return gr.update(choices=choices, value=None)

def optimize_playlist(selected_track, length, steps):
    if not selected_track:
        return "Please search and select a seed track.", None, None

    try:
        track_id = selected_track.rsplit("(", 1)[-1].replace(")", "").strip()
    except Exception:
        return "Could not parse track id.", None, None

    matches = df[df["track_id"] == track_id]
    if matches.empty:
        return "Track not found.", None, None

    seed_idx = df.index.get_loc(matches.index[0])
    seed = matches.iloc[0]

    # Run SA
    best, score = run_sa(seed_idx, int(length), steps=int(steps))
    _, comps = total_score(seed_idx, best)

    # Seed info
    info = f"""
**Seed Track**
**Name**: {seed['track_name']}
**Artists**: {seed['artists']}
**Genre**: {seed.get('track_genre', 'N/A')}
**Mood**: {seed.get('mood', 'N/A')}
**Popularity**: {seed['popularity']}

**Optimized Score**: {score:.3f}
- Smoothness: {comps['smoothness']:.3f}
- Similarity: {comps['similarity']:.3f}
- Diversity: {comps['diversity']:.3f}
- Mood consistency: {comps['mood']:.3f}
"""

    # Playlist table
    cols = ["track_name", "artists", "track_genre", "popularity"]
    if "mood" in df.columns:
        cols.append("mood")
    if "energy" in df.columns:
        cols.append("energy")
    if "tempo" in df.columns:
        cols.append("tempo")
    playlist_df = df.iloc[best][cols].copy()
    playlist_df.index = range(1, len(playlist_df) + 1)

    # Energy / tempo plot
    fig, ax1 = plt.subplots(figsize=(9, 3.5))
    x = np.arange(1, len(best) + 1)
    energy = df.iloc[best]["energy"].values
    tempo  = df.iloc[best]["tempo"].values
    ax1.plot(x, energy, "o-", color="steelblue", label="Energy")
    ax1.set_ylabel("Energy", color="steelblue")
    ax1.set_xlabel("Position")
    ax2 = ax1.twinx()
    ax2.plot(x, tempo, "s--", color="darkorange", label="Tempo")
    ax2.set_ylabel("Tempo (BPM)", color="darkorange")
    plt.title("Optimized Playlist – Energy & Tempo")
    fig.tight_layout()

    return info, playlist_df, fig

## 5. Build Gradio Interface


In [13]:
with gr.Blocks(title="Playlist Optimizer", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        """
        # Playlist Optimizer
        Content-based playlist optimization using **Simulated Annealing**.
        Balances smoothness, seed similarity, diversity, and mood consistency.
        """
    )

    with gr.Row():
        search_box = gr.Textbox(
            label="Search seed track or artist",
            placeholder="e.g. Blinding Lights, The Weeknd..."
        )
        search_btn = gr.Button("Search", variant="primary")

    track_dropdown = gr.Dropdown(label="Select seed track", choices=[], interactive=True)

    with gr.Row():
        length_slider = gr.Slider(6, 20, value=12, step=1, label="Playlist length")
        steps_slider  = gr.Slider(800, 5000, value=2500, step=100, label="SA steps (higher = better, slower)")

    run_btn = gr.Button("Optimize Playlist", variant="primary")

    seed_out = gr.Markdown(label="Seed & Scores")
    table_out = gr.Dataframe(label="Optimized Playlist")
    plot_out = gr.Plot(label="Energy & Tempo progression")

    search_btn.click(fn=search_tracks, inputs=search_box, outputs=track_dropdown)
    run_btn.click(
        fn=optimize_playlist,
        inputs=[track_dropdown, length_slider, steps_slider],
        outputs=[seed_out, table_out, plot_out]
    )



## 6. Launch


In [14]:
demo.launch(share=False)   # set share=True for a public link

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://19a30de0a08eaa75ab.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
